## import libraries

In [1]:
import ee
from dotenv import load_dotenv
import os
import ee
import pandas as pd
from pathlib import Path
import pickle

## initialize google earth engine

In [2]:
ee.Authenticate()

True

In [3]:
load_dotenv()  
project = os.getenv("PROJECT")
ee.Initialize(project=project)

## read datasets

In [4]:
root_embeddings = Path("_data/exports/embeddings_exports")
root_dataset = Path("_data/exports/crop_country_exports")

In [5]:
def build_files_dataframe(embeddings, dataset):
    rows = []

    folders = [dataset, embeddings]

    for folder in folders:
        for file in folder.glob("*"):
    
            if file.suffix not in [".csv", ".parquet"]:
                continue
            name = file.stem  
            parts = name.split("_")

            if "embedding" in parts:
                year = parts[-2]
                country = parts[-3]
                crop = "_".join(parts[:-3])
                file_type = "embedding"
            else:
                year = parts[-1]
                country = parts[-2]
                crop = "_".join(parts[:-2])
                file_type = "dataset"
            
            rows.append({
                "path": str(file),
                "crop": crop,
                "country": country,
                "year": int(year),
                "type": file_type
            })

    return pd.DataFrame(rows)

In [6]:
files = build_files_dataframe(root_embeddings, root_dataset)
files

,path,crop,country,year,type
0,_data/exports/crop_country_exports/maize_corn_...,maize_corn_popcorn,pt,2018,dataset
1,_data/exports/crop_country_exports/maize_corn_...,maize_corn_popcorn,pt,2019,dataset
2,_data/exports/crop_country_exports/potatoes_bg...,potatoes,bg,2018,dataset
3,_data/exports/crop_country_exports/winter_barl...,winter_barley,dk,2018,dataset
4,_data/exports/crop_country_exports/winter_barl...,winter_barley,be,2016,dataset
...,...,...,...,...,...
92,_data/exports/embeddings_exports/maize_corn_po...,maize_corn_popcorn,be,2019,embedding
93,_data/exports/embeddings_exports/winter_barley...,winter_barley,ee,2019,embedding
94,_data/exports/embeddings_exports/potatoes_at_2...,potatoes,at,2019,embedding
95,_data/exports/embeddings_exports/maize_corn_po...,maize_corn_popcorn,ie,2019,embedding


In [7]:
# pairing embeddings with datasets
df_datasets = files[files['type'] == 'dataset'].drop(columns=['type'])
df_embeddings = files[files['type'] == 'embedding'].drop(columns=['type'])

pairs = pd.merge(df_datasets, 
                 df_embeddings, 
                 on=['crop', 'country', 'year'], 
                 suffixes=('_dataset', '_embedding'))

pairs

,path_dataset,crop,country,year,path_embedding
0,_data/exports/crop_country_exports/maize_corn_...,maize_corn_popcorn,pt,2019,_data/exports/embeddings_exports/maize_corn_po...
1,_data/exports/crop_country_exports/winter_barl...,winter_barley,dk,2019,_data/exports/embeddings_exports/winter_barley...
2,_data/exports/crop_country_exports/potatoes_bg...,potatoes,bg,2019,_data/exports/embeddings_exports/potatoes_bg_2...
3,_data/exports/crop_country_exports/maize_corn_...,maize_corn_popcorn,bg,2019,_data/exports/embeddings_exports/maize_corn_po...
4,_data/exports/crop_country_exports/potatoes_pt...,potatoes,pt,2019,_data/exports/embeddings_exports/potatoes_pt_2...
5,_data/exports/crop_country_exports/potatoes_dk...,potatoes,dk,2019,_data/exports/embeddings_exports/potatoes_dk_2...
6,_data/exports/crop_country_exports/winter_barl...,winter_barley,bg,2019,_data/exports/embeddings_exports/winter_barley...
7,_data/exports/crop_country_exports/maize_corn_...,maize_corn_popcorn,at,2017,_data/exports/embeddings_exports/maize_corn_po...
8,_data/exports/crop_country_exports/potatoes_at...,potatoes,at,2017,_data/exports/embeddings_exports/potatoes_at_2...
9,_data/exports/crop_country_exports/maize_corn_...,maize_corn_popcorn,dk,2019,_data/exports/embeddings_exports/maize_corn_po...


In [8]:
datasets_dict = {}
embeddings_dict = {}

for _, row in pairs.iterrows():
    key = f"{row['crop']}_{row['country']}_{row['year']}"
    
    datasets_dict[key] = pd.read_csv(row['path_dataset'])
    embeddings_dict[key] = pd.read_parquet(row['path_embedding'])

print(f"Successfully loaded {len(datasets_dict)} pairs.")
print(f"Dictionaries keys are  {datasets_dict.keys()} pairs.")

Successfully loaded 22 pairs.
Dictionaries keys are  dict_keys(['maize_corn_popcorn_pt_2019', 'winter_barley_dk_2019', 'potatoes_bg_2019', 'maize_corn_popcorn_bg_2019', 'potatoes_pt_2019', 'potatoes_dk_2019', 'winter_barley_bg_2019', 'maize_corn_popcorn_at_2017', 'potatoes_at_2017', 'maize_corn_popcorn_dk_2019', 'maize_corn_popcorn_be_2019', 'potatoes_ie_2019', 'winter_barley_ee_2019', 'potatoes_at_2019', 'maize_corn_popcorn_at_2019', 'maize_corn_popcorn_ie_2019', 'potatoes_be_2019', 'potatoes_ee_2019', 'winter_barley_at_2019', 'winter_barley_ie_2019', 'winter_barley_be_2019', 'maize_corn_popcorn_ee_2019']) pairs.


In [9]:
datasets_dict['maize_corn_popcorn_at_2019'].head(2)

,country_id,year,hcat4_code,hcat4_name,long,lat
0,at,2019,3310106000,maize_corn_popcorn,15.985728,48.643581
1,at,2019,3310106000,maize_corn_popcorn,16.546970,47.979057


In [10]:
embeddings_dict['maize_corn_popcorn_at_2019'].head(2)

,tile_lon,tile_lat,pixel_row,pixel_col,crs,embedding,long_lat,crop,country_id,year
0,15.95,48.65,628,638,EPSG:32633,"[3.0008357, 1.5298377, 4.059954, 7.472669, 3.2...","[15.985727711616656, 48.64358116680461]",maize_corn_popcorn,at,2019
1,16.55,47.95,240,355,EPSG:32633,"[4.665599, 1.133074, 2.932662, 4.7989016, 3.33...","[16.546969834985113, 47.979056503789096]",maize_corn_popcorn,at,2019


## for testing reduced dictionaries

In [11]:
reduction_size = len(embeddings_dict['maize_corn_popcorn_pt_2019'])

In [12]:
datasets_dict_reduced = {name: df.iloc[:reduction_size] for name, df in datasets_dict.items()}
embeddings_dict_reduced = {name: df.iloc[:reduction_size] for name, df in embeddings_dict.items()}

## get point images

| Data Type   | Access Method                     | Description                                                |
|--------------|----------------------------------|------------------------------------------------------------|
| Dataset     | `datasets_dict[key].iloc[i]`     | The i-th row of tabular data (labels, etc.).              |
| Embeddings   | `embeddings_dict[key].iloc[i]`   | The i-th row of the embedding vector.                     |
| Satellite    | `gee_dict[key][i]['s2']`         | The full Sentinel-2 time series for that exact point.     |

In [13]:
def get_sits_data(lat, lon, year):
    point = ee.Geometry.Point([lon, lat])
    start = f'{year}-01-01'
    end = f'{year}-12-31'

    s1_bands = ['VV', 'VH']
    s1_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
              .filterBounds(point)
              .filterDate(start, end)
              .filter(ee.Filter.eq('instrumentMode', 'IW'))
              .select(s1_bands))
    
    s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    s2_col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterBounds(point)
              .filterDate(start, end)
              #.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 100))
              .select(s2_bands))
    
    s1_raw = s1_col.getRegion(point, 10).getInfo() # 10 is the resolution
    s2_raw = s2_col.getRegion(point, 10).getInfo() 

    return s1_raw, s2_raw

In [14]:
def to_df(raw_data):
    df = pd.DataFrame(raw_data[1:], columns=raw_data[0])
    df['date'] = pd.to_datetime(df['time'], unit='ms')
    return df.drop(columns=['id', 'time'])

In [20]:
# # for only year extraction
# from pathlib import Path
# import os

# def get_inventory(root_path):
#     inventory = []

#     for file_path in root_path.glob("*.csv"):
#         parts = file_path.stem.split('_')
        
#         if len(parts) > 3:
#             crop = "_".join(parts[:-2])
#             country = parts[-2]
#             year = int(parts[-1])

#         else:  
#             crop = parts[0]
#             country = parts[-2]
#             year = int(parts[-1])

#         inventory.append({
#             "crop": crop,
#             "country": country,
#             "year": year,
#             "path": file_path
#         })

#     df = pd.DataFrame(inventory)

#     if not df.empty:
#         df = df.sort_values(by=["crop", 'year']).reset_index(drop=True)

#     return df

# available_datasets = get_inventory(Path('_data/exports/crop_country_exports'))

# def filter_df(df, country):
#     return df[(df['country'] == country)]

# filtered_dataset = filter_df(available_datasets, 'at')
# filtered_dataset

# def create_dataset(df):
#     dataset = {}

#     for (crop, country, year), group in df.groupby(['crop', 'country', 'year']):
#         key = f"{crop}_{country}_{year}"
        
#         dataset[key] = pd.concat(
#             [pd.read_csv(p) for p in group['path']],
#             ignore_index=True
#         )
    
#     return dataset

# dataset_temporal_analysis = create_dataset(filtered_dataset)
# dataset_temporal_analysis.keys()

In [ ]:
keys = ['maize_corn_popcorn_at_2019', 'maize_corn_popcorn_be_2019', 'maize_corn_popcorn_bg_2019', 'maize_corn_popcorn_dk_2019', 'maize_corn_popcorn_ee_2019', 'maize_corn_popcorn_ie_2019', 'potatoes_at_2019', 'potatoes_be_2019', 'potatoes_bg_2019', 'potatoes_dk_2019', 'potatoes_ee_2019', 'potatoes_ie_2019']
datasets_dict.keys()

In [19]:
images_dict = {}

for key in dataset_temporal_analysis.keys(): # para analise temporal
    df_main = dataset_temporal_analysis[key]
# for key in keys: # ir fazendo só para alguns
#     df_main = datasets_dict[key]
# for key in datasets_dict.keys(): # todos os datasets
#     df_main = datasets_dict[key]
# for key in datasets_dict_reduced.keys(): # para datasets mais pequenos
#     df_main = datasets_dict_reduced[key]
    year = int(key.split('_')[-1])
    
    file_results = []
    
    print(f"Starting extraction for {key} ({len(df_main)} points)...")
    
    for idx, row in df_main.iterrows():
        lat, lon = row['lat'], row['long']
        
        try:
            s1_raw, s2_raw = get_sits_data(lat, lon, year)
            df_s1 = to_df(s1_raw)
            df_s2 = to_df(s2_raw)

            file_results.append({'s1': df_s1, 
                                 's2': df_s2, 
                                 'point_coord': (lon, lat)})
            
        except Exception as e:
            print(f"Error at row {idx} in {key}: {e}")
            file_results.append(None) 
            
    images_dict[key] = file_results

Starting extraction for maize_corn_popcorn_at_2015 (500 points)...
Error at row 49 in maize_corn_popcorn_at_2015: ImageCollection.getRegion: Error in map(ID=S1A_IW_GRDH_1SDH_20150708T052614_20150708T052633_006715_008FD2_178B):
Image.select: Band pattern 'VV' did not match any bands. Available bands: [HH, HV, angle]
Error at row 206 in maize_corn_popcorn_at_2015: ImageCollection.getRegion: No bands in collection.
Error at row 210 in maize_corn_popcorn_at_2015: ImageCollection.getRegion: No bands in collection.
Error at row 271 in maize_corn_popcorn_at_2015: ImageCollection.getRegion: No bands in collection.
Error at row 280 in maize_corn_popcorn_at_2015: ImageCollection.getRegion: No bands in collection.
Error at row 283 in maize_corn_popcorn_at_2015: ImageCollection.getRegion: No bands in collection.
Error at row 290 in maize_corn_popcorn_at_2015: ImageCollection.getRegion: No bands in collection.
Error at row 329 in maize_corn_popcorn_at_2015: ImageCollection.getRegion: No bands in co

KeyboardInterrupt: 

In [ ]:
images_dict.keys()

In [ ]:
# print(f"type(images_dict['winter_barley_dk_2019']) is a {type(images_dict['winter_barley_dk_2019'])}")
# print(f"type(images_dict['winter_barley_dk_2019']) has size {len(images_dict['winter_barley_dk_2019'])}, for each point")
# print(f"type(images_dict['winter_barley_dk_2019']) keys are {images_dict['winter_barley_dk_2019'][0].keys()}")

In [ ]:
# images_dict['winter_barley_dk_2019'][0]['point_coord']

## export results

In [ ]:
images_output_dir = Path("_data/exports/image_points_exports")
images_output_dir.mkdir(parents=True, exist_ok=True)

def save_gee_data(data_dict, folder):
    for key, content in data_dict.items():
        file_path = folder / f"{key}_image.pkl"
        with open(file_path, 'wb') as f:
            pickle.dump(content, f)
    print(f"Successfully saved {len(data_dict)} files to {folder}")

save_gee_data(images_dict, images_output_dir)

In [ ]:
# to load later
with open('_data/exports/image_points_exports/potatoes_dk_2019_image.pkl', 'rb') as f:
    ooo = pickle.load(f)

ooo[0]['point_coord']